# Visualisations avancées : cartes de tirs

Ce notebook présente des visualisations avancées de cartes de tirs pour une équipe donnée de la LNH et pour une année et saison données. 

## 1. Normaliser les tirs

Pour normaliser les tirs, nous utilisons la fonction normalize_shots, qui exploite le fichier CSV généré lors de la tâche 3 afin de produire un nouveau fichier CSV contenant les coordonnées normalisées ainsi que les coordonnées en pixels (pixel_x et pixel_y), adaptées à l'échelle de la ShotMap finale.

In [ ]:
from ift6758.features.shot_features import (
    normalize_shots,
    shots_per_hour,
    extract_shots,
)
import ipywidgets as widgets
from ipywidgets import interact

Ici j'ai mis des valeurs temporaires pour tester la fonction.

In [ ]:
# Décommenté si pas déjà fait en Q4
normalize_shots(565, 681) 

## 2. Statistiques agrégées des emplacements de tirs

On cherche à calculer pour chaque région du terrain le nombre de tir et son taux de tirs moyen par heure de la ligue, pour cela on récupère tous les tirs de la ligue pour une saison donnée. Puis on divise le terrain par région et on compte le nombre de tirs et le taux de tirs par région. Le tableau retourné montre pour chaque région son résultat.

In [ ]:
saisons = [20162017, 20172018, 20182019, 20192020, 20202021, 20212022, 20222023, 20232024]
@interact(
    #Widget pour selectionner la saison
    season_id=widgets.Dropdown(options=saisons, value=saisons[0], description="Saison :"),
)
def interactive_game_id(season_id):
    df = shots_per_hour(10,5, season_id)
    display(df)

## 3 & 4. Statistiques agrégées des emplacements de tirs par équipe selon un binning appropri.

On cherche à calculer pour chaque région du terrain le nombre de tir et son taux de tirs moyen par heure d'une équipe par rapport au taux de la ligue, pour cela on récupère tous les tirs de la ligue pour une saison donnée et une équipe donnée. Puis on divise le terrain par région et on compte le nombre de tirs et le taux de tirs par région. Le tableau retourné montre pour chaque région son résultat du taux de la ligue par rapport à l'équipe voulue. Pour choisir un "binning" approprié nous allons d'abord découpé la demi-glace en portion fines soit 50 bins en X et 42 bins en y ce qui donne à peu près 2 pieds d'intervalles. Puis nous allons ajouter un noyau gaussien pour lisser. Voici un exemple simple pour voir le taux d'excès d'une équipe en particulier.

In [ ]:
# Dictionnaire de team_id 
equipes = {
    "Colorado Avalanche": 21,
    "Tampa Bay Lightning": 14,
    "Buffalo Sabres": 7,
    "Montréal Canadiens": 8,
    "Toronto Maples Leafs": 10
}

saisons = [20162017, 20172018, 20182019, 20192020, 20202021]
@interact(
    #Widget pour selectionner la saison
    season_id=widgets.Dropdown(options=saisons, value=saisons[0], description="Saison :"),
    team_name=widgets.Dropdown(options=list(equipes.keys()), value="Colorado Avalanche", description= "Équipe :")
)
def interactive_excess_rate(season_id, team_name):
    df_excess_rate = shots_per_hour(10, 5, season_id, team=equipes[team_name])
    if df_excess_rate is not None:
        display(df_excess_rate.head(10))

Maintenant il faut écrire une fonction qui permet de prendre une saison en particulière et de générer un graphique intéractif sur le choix des équipes. Ce graphique prendra les données de taux calculé aux étapes 1-2-3 puis apliquera un lissage gaussien et l'affichera sur un graphique intéractif. Après avoir essayé plusieurs combinaisopns de x_bins et y_bins nous en déduisons que 25 cases en x et 21 cases en y est le compromis parfait. Ça donne des cases d'environs 4pi*4pi et cela permet de voir les tendances générales de tirs de chaque équipe, en essayant le double soit 50*42 on se noyait dans les détails inutiles. Et si on mettais moins de bins alors, il n'y avait pas assez de détails. De plus, avec un noyau gaussien de sigma 1.5, cela permet de lisser les résultats et de couper le bruit. Ainsi, les contours sont clairs et il est plus facile de voir les tendances sur le graphique.

In [ ]:
import plotly.graph_objects as go 
from pathlib import Path
import numpy as np
import pandas as pd
from ift6758.data_acquisition import (
    draw_semi_rink,
)
from ift6758.features.shot_features import get_season_excess_shots


x_bins = 25
y_bins = 21

# L'implémentation du dropdown est inspiré de https://plotly.com/python/dropdowns/ et pour le contour https://plotly.com/python/contour-plots/
def make_shot_graph(season):
    mat_teams, xedges, yedges = get_season_excess_shots(season, x_bins, y_bins)

    figure = go.Figure(
        layout = go.Layout(        
        xaxis=dict(range=[0, 100]),
        yaxis=dict(range=[-42.5, 42.5]),
        width=800,
        height=700)
    )
    rink = draw_semi_rink(figure)
    # Car sinon rink écrasait les axes
    figure.update_xaxes(title="Longueur de la patinoire (ft)")
    figure.update_yaxes(title="Distance du centre de la patinoire (ft)")

    drop_down_buttons = []
    teams = list(mat_teams.keys())

    # On veut le centre de nos cases
    centers_x = np.zeros(x_bins)
    centers_y = np.zeros(y_bins)
    # On peut probablement le faire plus rapidement sans boucle, mais c'est plus clair
    for i in range(x_bins):
        centers_x[i] = (xedges[i] + xedges[i+1]) / 2

    for i in range(y_bins):
        centers_y[i] = (yedges[i] + yedges[i+1]) / 2
    
    # Pour chaque équipe, on a rajoute la trace de leur taux
    for i, team  in enumerate(teams):
        mat = mat_teams[team]
        # On doit ajouter les traces en gardant une seule visible
        Is_visible = i == 0
        figure.add_trace(go.Contour(z = mat, x = centers_x, y=centers_y, colorscale='RdBu_r', zmid=0, contours=dict(
                start=-0.07, # On se donne assez de marge
                end=0.07,
                size=0.01), 
        opacity=0.6, visible=Is_visible, name=team,
        colorbar=dict(title=dict(text="Tirs excédentaires<br>par heure", side="right"))))

        # La logique du bouton est de rendre visible la trace si elle est choisie
        # On va utiliser dans la method "update" pour modifier les data et le layout
        # On doit passer en argument la liste de booléen en mettant true à l'indice de la trace sélectionnée
        # Comme fait dans la section Update Dropdown de la documentation
        visible_traces = [False]*len(teams)
        # Dans ce cas, si le bouton est choisi c'est cette équipe qui sera choisie
        visible_traces[i] = True
        drop_down_buttons.append(
            dict(
                args=[{"visible": visible_traces},
                      {"title": f"Taux de tirs excédentaire par heure par rapport à la ligue: {teams[i]} ({season})"}],
                      label = team,
                      method = "update"
            )
        )


    # On ajoute le style et les buttons
    # Les attributs de styles viennent des exemples de la documentation
    figure.update_layout(
        updatemenus=[
            dict(
                buttons=drop_down_buttons,
                direction="up",
                pad={"r": 5, "t": 5},
                showactive=True,
                x=-0.02,
                xanchor="left",
                y=-0.01,
                yanchor="bottom"
            )
        ],
        title = f"Taux de tirs excédentaire par heure par rapport à la ligue: {teams[0]} ({season})",
    )

    figure.show()
    output_folder = Path.cwd().parent/"figures"
    output_folder.mkdir(parents=True, exist_ok=True)

    # Puisque les documentations et les exemples que j'ai utilisé utilisait plotly.graph_objects
    # et non plotly.express, j'ai dû demander de l'aide pour trouver comment, voir AI logs.
    figure.write_html(output_folder/f"shot_map_{season}.html", include_plotlyjs='cdn', full_html=False)


    
    
make_shot_graph(20182019)



Nous allons maintenant imprimer les graphiques pour chaque saison :

In [ ]:
for season in saisons:
    make_shot_graph(season)